# Community responses at a low streptomycin concentration

The antibiotic response of a landscape is its composition at a given dose minus its
composition without antibiotic at the same carbon level. This notebook compares responses
between focal-species histories, with emphasis on 2 ug/ml, below the ancestral MICs of all
members of the 16-species community.

Outputs: Figure 4, Figure S7, Tables S4-S7, S8-S9, and in `results/statistics/` the
species detection frequencies, the whole-community version of the species contrasts and
the pseudocount sensitivity of log-ratio contrasts.

In [1]:
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import spearmanr

%run 00_setup_and_functions.ipynb

## Log-ratio responses
Species-level CLR basis: the taxa detected in more than half of the 16-species samples.
The non-focal subcomposition removes *A. caviae* and recloses the other 15 species, so that
its response is not an arithmetic consequence of the focal species' change.

In [2]:
dclr_rep, dclr_cols = whole_community_clr_response()
nf_stressed, nf_dclr_cols = clr_response(NONFOCAL_SPECIES, "nf_clr")
species_contrasts_clr = species_clr_history_contrasts(dclr_rep)

## Table S4: whole-community response
Exact PERMANOVA in Euclidean CLR (Aitchison) space on the carbon-averaged response of each landscape.

In [3]:
rows = []
for richness, anc, evo in RICHNESS_PAIRS:
    for dose in NONZERO_DOSES:
        sub = dclr_rep[dclr_rep.community.isin([anc, evo]) & (dclr_rep.streptomycin == dose)]
        F, R2, p, _ = exact_permanova(sub[dclr_cols].to_numpy(float), (sub.community == evo).astype(int).to_numpy())
        rows.append({"richness": int(richness), "streptomycin": dose, "pseudo_F": F, "R2": R2, "exact_P": p,
                     "n_landscapes_per_history": 8})
table_s4 = pd.DataFrame(rows)
table_s4.to_csv(TABLE_DIR / "Table_S4_compositional_response_PERMANOVA.csv", index=False)
table_s4

,richness,streptomycin,pseudo_F,R2,exact_P,n_landscapes_per_history
0,3,2.0,4.631758,0.248595,0.001088,8
1,3,20.0,1032.985629,0.986628,0.000155,8
2,3,200.0,325.740951,0.958792,0.000155,8
3,16,2.0,14.465819,0.508182,0.000155,8
4,16,20.0,64.845139,0.822437,0.000155,8
5,16,200.0,53.785690,0.793467,0.000155,8


## Focal-species response at 2 ug/ml (Tables S5 and S8)
Change in *A. caviae* relative abundance from 0 to 2 ug/ml at the same carbon level. Table S5:
carbon-averaged contrast (Holm across the two richness levels), distance between the history
centroids of the 16-level response landscapes, and the mean absolute history difference across
carbon levels. Table S8: contrast at each carbon level, Benjamini-Hochberg across levels.

In [4]:
base = (sample_compositions.loc[sample_compositions.streptomycin == 0, ["community", "replicate", "carbon_g_L", FOCAL_SPECIES]]
        .rename(columns={FOCAL_SPECIES: "focal_no_antibiotic"}))
focal_lowdose = (sample_compositions.loc[sample_compositions.streptomycin == LOW_DOSE, ["community", "replicate", "carbon_g_L", FOCAL_SPECIES]]
                 .merge(base, on=["community", "replicate", "carbon_g_L"], how="left", validate="one_to_one"))
focal_lowdose["delta_focal_abundance"] = focal_lowdose[FOCAL_SPECIES] - focal_lowdose["focal_no_antibiotic"]


def focal_response_matrix(community):
    """Landscape x carbon matrix of the focal response."""
    return (focal_lowdose.loc[focal_lowdose.community == community]
            .pivot(index="replicate", columns="carbon_g_L", values="delta_focal_abundance")
            .reindex(columns=CARBON_VALUES).to_numpy(float))


landscape_means = focal_lowdose.groupby(["community", "replicate"])["delta_focal_abundance"].mean()
s5_rows, s13_rows = [], []
for richness, anc, evo in RICHNESS_PAIRS:
    A, E = focal_response_matrix(anc), focal_response_matrix(evo)
    x, y = landscape_means[anc].to_numpy(float), landscape_means[evo].to_numpy(float)
    diff, p = exact_mean_difference_test(x, y)
    lo, hi = bootstrap_diff_ci(x, y, seed=RANDOM_SEED + 12000 + int(richness))
    distance, p_distance = exact_centroid_distance_test(A, E)
    obs_abs = float(np.abs(E.mean(0) - A.mean(0)).mean())
    pooled = np.vstack([A, E])
    null_abs = np.array([np.abs(pooled[m].mean(0) - pooled[~m].mean(0)).mean() for m in independent_masks(16, 8).astype(bool)])
    s5_rows.append({"richness": richness, "ancestral_mean_delta_abundance": x.mean(), "evolved_mean_delta_abundance": y.mean(),
                    "evolved_minus_ancestral": diff, "ci95_low": lo, "ci95_high": hi, "p_exact": p,
                    "n_ancestral": len(x), "n_evolved": len(y),
                    "response_landscape_centroid_distance": distance, "p_exact_response_landscape": p_distance,
                    "mean_abs_history_difference": obs_abs, "p_exact_mean_abs_difference": float((null_abs >= obs_abs - 1e-12).mean())})
    family = []
    for k, carbon in enumerate(CARBON_VALUES):
        d, pk = exact_mean_difference_test(A[:, k], E[:, k])
        family.append({"richness": richness, "carbon_g_L": carbon, "ancestral_mean_delta_abundance": A[:, k].mean(),
                       "evolved_mean_delta_abundance": E[:, k].mean(), "evolved_minus_ancestral": d, "p_exact": pk})
    for r, q in zip(family, bh([r["p_exact"] for r in family])):
        r["q_bh_across_carbon"] = q
    s13_rows += family

table_s5 = pd.DataFrame(s5_rows)
table_s5.insert(table_s5.columns.get_loc("n_evolved") + 1, "p_holm_across_richness", holm(table_s5.p_exact))
table_s5.to_csv(TABLE_DIR / "Table_S5_lowdose_focal_response.csv", index=False)
table_s8 = pd.DataFrame(s13_rows)
table_s8.to_csv(TABLE_DIR / "Table_S8_lowdose_focal_response_by_carbon.csv", index=False)
table_s5

,richness,ancestral_mean_delta_abundance,evolved_mean_delta_abundance,evolved_minus_ancestral,ci95_low,ci95_high,p_exact,n_ancestral,n_evolved,p_holm_across_richness,response_landscape_centroid_distance,p_exact_response_landscape,mean_abs_history_difference,p_exact_mean_abs_difference
0,3,-0.017794,0.051573,0.069367,0.058530,0.080027,0.000155,8,8,0.000311,0.541077,0.000155,0.080319,0.000155
1,16,-0.071615,-0.042380,0.029236,0.006021,0.052561,0.037451,8,8,0.037451,1.110441,0.000155,0.186023,0.000155


## Table S6: focal and non-focal responses at 2 ug/ml in the 16-species community
Across carbon levels, the signed history effect on the focal response is related to the
Aitchison distance between the history centroids of the non-focal response (Spearman).
History labels are permuted across the 16 landscapes, keeping each landscape's carbon series
intact; the CI and the regression band in Figure 4C resample landscapes within history.
The whole non-focal response landscape (16 carbon levels x 15 CLR coordinates) is compared
between histories by the exact centroid-distance test.

In [5]:
nf_lowdose = nf_stressed.loc[nf_stressed.streptomycin == LOW_DOSE]


def nonfocal_response_cube(community):
    """Landscape x carbon x CLR-coordinate array of the non-focal response."""
    sub = nf_lowdose.loc[nf_lowdose.community == community]
    return np.asarray([sub.loc[sub.replicate == rep].set_index("carbon_g_L").reindex(CARBON_VALUES)[nf_dclr_cols].to_numpy(float)
                       for rep in sorted(sub.replicate.unique())], float)


Fa, Fe = focal_response_matrix("ANC_SIXTEEN"), focal_response_matrix("EVO_SIXTEEN")
Na, Ne = nonfocal_response_cube("ANC_SIXTEEN"), nonfocal_response_cube("EVO_SIXTEEN")
focal_contrast = Fe.mean(axis=0) - Fa.mean(axis=0)
nonfocal_distance = np.linalg.norm(Ne.mean(axis=0) - Na.mean(axis=0), axis=1)
rho = float(spearmanr(focal_contrast, nonfocal_distance).statistic)

F_pool, N_pool = np.vstack([Fa, Fe]), np.concatenate([Na, Ne], axis=0)
null_rho = []
for anc_idx in combinations(range(16), 8):
    anc_idx = np.asarray(anc_idx, int)
    evo_idx = np.setdiff1d(np.arange(16), anc_idx)
    null_rho.append(spearmanr(F_pool[evo_idx].mean(0) - F_pool[anc_idx].mean(0),
                              np.linalg.norm(N_pool[evo_idx].mean(0) - N_pool[anc_idx].mean(0), axis=1)).statistic)
p_rho = float(np.mean(np.abs(np.asarray(null_rho)) >= abs(rho) - 1e-15))

rng = np.random.default_rng(RANDOM_SEED + 13000)
x_grid = np.linspace(focal_contrast.min(), focal_contrast.max(), 200)
boot_rho = np.empty(10000); boot_line = np.empty((10000, len(x_grid)))
for b in range(10000):
    ia, ie = rng.integers(0, 8, 8), rng.integers(0, 8, 8)
    fc = Fe[ie].mean(0) - Fa[ia].mean(0)
    nd = np.linalg.norm(Ne[ie].mean(0) - Na[ia].mean(0), axis=1)
    boot_rho[b] = spearmanr(fc, nd).statistic
    boot_line[b] = np.polyval(np.polyfit(fc, nd, 1), x_grid) if np.ptp(fc) > 0 else np.nan
rho_ci = np.nanquantile(boot_rho, [0.025, 0.975])
line = np.polyval(np.polyfit(focal_contrast, nonfocal_distance, 1), x_grid)
line_lo, line_hi = np.nanquantile(boot_line, [0.025, 0.975], axis=0)

landscape_distance, landscape_p = exact_centroid_distance_test(Na.reshape(8, -1), Ne.reshape(8, -1))
table_s6 = pd.DataFrame([{
    "streptomycin": LOW_DOSE,
    "spearman_rho_signed_focal_contrast_vs_nonfocal_divergence": rho,
    "bootstrap95_low_signed": rho_ci[0], "bootstrap95_high_signed": rho_ci[1],
    "exact_randomization_p_signed_coupling": p_rho,
    "nonfocal_landscape_rms_centroid_separation": landscape_distance / np.sqrt(N_CARBON),
    "nonfocal_landscape_exact_p": landscape_p,
    "n_biological_replicates_per_history": 8, "n_glucose_levels": N_CARBON}])
table_s6.to_csv(TABLE_DIR / "Table_S6_lowdose_focal_nonfocal_coupling.csv", index=False)
table_s6

,streptomycin,spearman_rho_signed_focal_contrast_vs_nonfocal_divergence,bootstrap95_low_signed,bootstrap95_high_signed,exact_randomization_p_signed_coupling,nonfocal_landscape_rms_centroid_separation,nonfocal_landscape_exact_p,n_biological_replicates_per_history,n_glucose_levels
0,2.0,0.705882,0.45,0.797059,0.011344,9.093393,0.000155,8,16


## Species-level contrasts at 2 ug/ml (Tables S7 and S9)
Table S9 (Figure 4D): change in relative abundance from 0 to 2 ug/ml, carbon-averaged within
each landscape. *A. caviae* is measured in the whole community, the other species within the
non-focal subcomposition. The *A. caviae* contrast is the focal test of Table S5; the 15
non-focal species are Benjamini-Hochberg corrected together. Relative abundance needs no zero
replacement, unlike log-ratio coordinates of taxa that are often undetected.
Table S7: the log-ratio (CLR) version for the non-focal species of the CLR basis.

In [6]:
rel = sample_compositions[["community", "replicate", "carbon_g_L", "streptomycin"]].copy()
whole = sample_compositions[SPECIES].to_numpy(float)
whole = whole / whole.sum(axis=1, keepdims=True)
nonfocal = sample_compositions[NONFOCAL_SPECIES].to_numpy(float)
nonfocal = nonfocal / nonfocal.sum(axis=1, keepdims=True)
for j, sp in enumerate(SPECIES):
    rel[f"whole_{sp}"] = whole[:, j]
for j, sp in enumerate(NONFOCAL_SPECIES):
    rel[f"rel_{sp}"] = nonfocal[:, j]
rel[f"rel_{FOCAL_SPECIES}"] = rel[f"whole_{FOCAL_SPECIES}"]
rel_cols = [f"rel_{sp}" for sp in SPECIES] + [f"whole_{sp}" for sp in SPECIES]
low = rel.loc[rel.streptomycin == LOW_DOSE].merge(
    rel.loc[rel.streptomycin == 0, ["community", "replicate", "carbon_g_L"] + rel_cols].rename(columns={c: f"base_{c}" for c in rel_cols}),
    on=["community", "replicate", "carbon_g_L"], how="left", validate="one_to_one")
for c in rel_cols:
    low[f"d_{c}"] = low[c] - low[f"base_{c}"]
rel_rep = low.groupby(["community", "replicate"])[[f"d_{c}" for c in rel_cols]].mean().reset_index()

rows = []
for sp in SPECIES:
    x = rel_rep.loc[rel_rep.community == "ANC_SIXTEEN", f"d_rel_{sp}"].to_numpy(float)
    y = rel_rep.loc[rel_rep.community == "EVO_SIXTEEN", f"d_rel_{sp}"].to_numpy(float)
    diff, p = exact_mean_difference_test(x, y)
    lo, hi = bootstrap_diff_ci(x, y, seed=RANDOM_SEED + 9100 + SPECIES.index(sp))
    rows.append({"species": sp, "species_name": SPECIES_NAMES[sp],
                 "basis": "whole community" if sp == FOCAL_SPECIES else "non-focal subcomposition",
                 "ancestral_mean_delta_rel": x.mean(), "evolved_mean_delta_rel": y.mean(),
                 "evolved_minus_ancestral_delta_rel": diff, "ci95_low": lo, "ci95_high": hi, "p_exact": p})
species_contrasts = pd.DataFrame(rows)
is_focal = species_contrasts.species == FOCAL_SPECIES
species_contrasts["q_bh_across_15_nonfocal"] = np.nan
species_contrasts.loc[~is_focal, "q_bh_across_15_nonfocal"] = bh(species_contrasts.loc[~is_focal, "p_exact"])
species_contrasts["p_holm_focal_primary"] = np.nan
species_contrasts.loc[is_focal, "p_holm_focal_primary"] = float(table_s5.loc[table_s5.richness == "16", "p_holm_across_richness"].iloc[0])
species_contrasts["p_adjusted"] = species_contrasts.q_bh_across_15_nonfocal.fillna(species_contrasts.p_holm_focal_primary)
species_contrasts.drop(columns=["species", "p_adjusted"]).to_csv(TABLE_DIR / "Table_S9_lowdose_species_relative_abundance_contrasts.csv", index=False)

table_s7 = species_contrasts_clr.loc[(species_contrasts_clr.streptomycin == LOW_DOSE) & (species_contrasts_clr.species != FOCAL_SPECIES),
                                     ["species_name", "evolved_minus_ancestral_delta_clr", "ci95_low", "ci95_high", "p_exact", "q_bh_across_species"]]
table_s7.to_csv(TABLE_DIR / "Table_S7_species_lowdose_history_contrasts.csv", index=False)
species_contrasts

,species,species_name,basis,ancestral_mean_delta_rel,evolved_mean_delta_rel,evolved_minus_ancestral_delta_rel,ci95_low,ci95_high,p_exact,q_bh_across_15_nonfocal,p_holm_focal_primary,p_adjusted
0,HAMBI_0006,Pseudomonas putida HAMBI 6,non-focal subcomposition,-8.586156e-02,-0.032426,0.053435,0.044962,0.062128,0.000155,0.001166,NaN,0.001166
1,HAMBI_0105,Agrobacterium tumefaciens HAMBI 105,non-focal subcomposition,1.933679e-08,-0.000044,-0.000044,-0.000153,0.000071,0.529138,0.721551,NaN,0.721551
2,HAMBI_0403,Comamonas testosteroni HAMBI 403,non-focal subcomposition,-7.706755e-04,-0.000442,0.000328,-0.000075,0.000718,0.150583,0.282343,NaN,0.282343
3,HAMBI_1287,Citrobacter koseri HAMBI 1287,non-focal subcomposition,-8.951993e-02,-0.044432,0.045088,0.017605,0.071927,0.009479,0.028438,NaN,0.028438
4,HAMBI_1292,Morganella morganii HAMBI 1292,non-focal subcomposition,2.157395e-04,0.000226,0.000010,-0.000067,0.000094,0.816162,0.941725,NaN,0.941725
5,HAMBI_1299,Kluyvera intermedia HAMBI 1299,non-focal subcomposition,0.000000e+00,0.000002,0.000002,0.000000,0.000007,1.000000,1.000000,NaN,1.000000
6,HAMBI_1842,Sphingobium yanoikuyae HAMBI 1842,non-focal subcomposition,4.965476e-03,-0.000120,-0.005086,-0.005883,-0.004208,0.000155,0.001166,NaN,0.001166
7,HAMBI_1896,Sphingobacterium spiritivorum HAMBI 1896,non-focal subcomposition,-7.006144e-03,-0.024737,-0.017731,-0.029117,-0.004914,0.020357,0.045954,NaN,0.045954
8,HAMBI_1972,Aeromonas caviae HAMBI 1972,whole community,-7.161532e-02,-0.042380,0.029236,0.006039,0.052633,0.037451,NaN,0.037451,0.037451
9,HAMBI_1977,Pseudomonas chlororaphis HAMBI 1977,non-focal subcomposition,1.020399e-01,0.017282,-0.084758,-0.127172,-0.042187,0.002797,0.010490,NaN,0.010490


Whole-community version of the relative-abundance contrasts for all 16 species (every species
on the same denominator; Benjamini-Hochberg across 16 species).

In [7]:
rows = []
for sp in SPECIES:
    x = rel_rep.loc[rel_rep.community == "ANC_SIXTEEN", f"d_whole_{sp}"].to_numpy(float)
    y = rel_rep.loc[rel_rep.community == "EVO_SIXTEEN", f"d_whole_{sp}"].to_numpy(float)
    diff, p = exact_mean_difference_test(x, y)
    rows.append({"species_name": SPECIES_NAMES[sp], "evolved_minus_ancestral_delta_rel": diff, "p_exact": p})
species_contrasts_whole = pd.DataFrame(rows)
species_contrasts_whole["q_bh_across_16_species"] = bh(species_contrasts_whole.p_exact)
species_contrasts_whole.to_csv(STATS_DIR / "lowdose_species_contrasts_whole_community.csv", index=False)
species_contrasts_whole

,species_name,evolved_minus_ancestral_delta_rel,p_exact,q_bh_across_16_species
0,Pseudomonas putida HAMBI 6,0.031361,0.000155,0.000622
1,Agrobacterium tumefaciens HAMBI 105,-0.000029,0.028438,0.065002
2,Comamonas testosteroni HAMBI 403,0.000168,0.013209,0.040611
3,Citrobacter koseri HAMBI 1287,0.029223,0.015229,0.040611
4,Morganella morganii HAMBI 1292,-0.000043,0.111422,0.162068
5,Kluyvera intermedia HAMBI 1299,0.000002,1.000000,1.000000
6,Sphingobium yanoikuyae HAMBI 1842,-0.004849,0.000155,0.000622
7,Sphingobacterium spiritivorum HAMBI 1896,-0.002771,0.052991,0.084786
8,Aeromonas caviae HAMBI 1972,0.029236,0.037451,0.066580
9,Pseudomonas chlororaphis HAMBI 1977,-0.042144,0.036519,0.066580


## Detection frequencies and pseudocount sensitivity of log-ratio contrasts
Fraction of 16-species samples in which each species was detected, overall and by history
and dose; and the 2 ug/ml CLR history contrasts of the 16-species community recomputed with
three zero-replacement constants on the species-level basis and on all 16 species.

In [8]:
sixteen = sample_compositions.loc[sample_compositions.community.isin(["ANC_SIXTEEN", "EVO_SIXTEEN"])]
detection = (sixteen[["community", "streptomycin"] + SPECIES].melt(id_vars=["community", "streptomycin"], var_name="species")
             .assign(detected=lambda d: d.value > 0)
             .pivot_table(index="species", columns=["community", "streptomycin"], values="detected", aggfunc="mean"))
detection.columns = [f"{c}_{s:g}ugml" for c, s in detection.columns]
detection.insert(0, "all_16_species_samples", [SPECIES_PREVALENCE[s] for s in detection.index])
detection.insert(0, "in_CLR_basis", [s in CLR_SPECIES for s in detection.index])
detection.insert(0, "species_name", [SPECIES_NAMES[s] for s in detection.index])
detection.reset_index().to_csv(STATS_DIR / "species_detection_16_species.csv", index=False)

rows = []
for basis_name, basis in [("species-level CLR basis", CLR_SPECIES), ("all 16 species", SPECIES)]:
    for pseudocount in [1e-6, 1e-4, 1e-3]:
        cols = [f"clr_{s}" for s in basis]
        df = sixteen[["community", "replicate", "carbon_g_L", "streptomycin"]].copy()
        df[cols] = clr(sixteen[basis].to_numpy(float), pseudocount)
        response = (df.loc[df.streptomycin == LOW_DOSE].set_index(["community", "replicate", "carbon_g_L"])[cols]
                    - df.loc[df.streptomycin == 0].set_index(["community", "replicate", "carbon_g_L"])[cols])
        rep = response.groupby(level=["community", "replicate"]).mean()
        tests = [exact_mean_difference_test(rep.loc["ANC_SIXTEEN"][c], rep.loc["EVO_SIXTEEN"][c]) for c in cols]
        q = bh([p for _, p in tests])
        rows.append({"basis": basis_name, "pseudocount": pseudocount,
                     "focal_contrast_delta_clr": tests[basis.index(FOCAL_SPECIES)][0],
                     "n_species_q_below_0.05": int((q < 0.05).sum()), "n_species": len(basis)})
pseudocount_sensitivity = pd.DataFrame(rows)
pseudocount_sensitivity.to_csv(STATS_DIR / "clr_pseudocount_sensitivity.csv", index=False)
pseudocount_sensitivity

,basis,pseudocount,focal_contrast_delta_clr,n_species_q_below_0.05,n_species
0,species-level CLR basis,0.000001,-0.935733,3,9
1,species-level CLR basis,0.000100,-0.742798,4,9
2,species-level CLR basis,0.001000,-0.553906,6,9
3,all 16 species,0.000001,-0.932055,4,16
4,all 16 species,0.000100,-0.693753,10,16
5,all 16 species,0.001000,-0.501236,11,16


## Figure 4
A,B, change in *A. caviae* relative abundance from 0 to 2 ug/ml along the carbon gradient (Table S8); C, history effect on the focal response versus divergence of the non-focal response across carbon levels (Table S6); D, species-level contrasts (Table S9).

In [9]:
contrast = species_contrasts.set_index("species")
order = list(contrast.evolved_minus_ancestral_delta_rel.sort_values().index)

fig = plt.figure(figsize=(183 * MM, 154 * MM))
gs = fig.add_gridspec(2, 3, width_ratios=[1.0, 1.0, 1.62], height_ratios=[1, 1], wspace=0.90, hspace=0.60)


# A, B: focal response along carbon (history means and 95% CIs across the eight landscapes);
# asterisks mark carbon levels with q < 0.05 (Table S8).
def focal_curve(ax, anc, evo, richness, title, letter):
    panel_label(ax, letter)
    ax.axhline(0, color=REFERENCE_GREY, lw=0.6, zorder=1)
    for community, color, label in [(anc, ANCESTRAL_COLOR, "Ancestral"), (evo, EVOLVED_COLOR, "Resistance-evolved")]:
        mat = focal_response_matrix(community)
        mean = mat.mean(axis=0)
        ci = 1.96 * mat.std(axis=0, ddof=1) / np.sqrt(mat.shape[0])
        ax.plot(CARBON_VALUES, mean, marker="o", ms=2.8, lw=1.15, color=color, label=label, zorder=3)
        ax.fill_between(CARBON_VALUES, mean - ci, mean + ci, color=color, alpha=0.16, linewidth=0, zorder=2)
    significant = table_s8.loc[(table_s8.richness == richness) & (table_s8.q_bh_across_carbon < 0.05), "carbon_g_L"]
    ytop = ax.get_ylim()[1]
    for carbon in significant:
        ax.text(carbon, ytop, "*", ha="center", va="bottom", fontsize=7, fontweight="bold")
    ax.set_ylim(ax.get_ylim()[0], ytop + 0.08 * (ytop - ax.get_ylim()[0]))
    ax.set_xscale("log")
    ax.set_xlabel(CARBON_AXIS_LABEL)
    ax.set_ylabel(r"Change in $\it{A.\ caviae}$ relative abundance" "\n(0 to 2 µg ml$^{-1}$ streptomycin)")
    ax.set_title(title, fontsize=6.6, pad=9)


focal_curve(fig.add_subplot(gs[0, 0]), "ANC_THREE", "EVO_THREE", "3", "3 species", "a")
focal_curve(fig.add_subplot(gs[0, 1]), "ANC_SIXTEEN", "EVO_SIXTEEN", "16", "16 species", "b")

# C: history effect on the focal response against divergence of the non-focal response, across
# carbon levels; linear fit with a bootstrap band as a visual guide.
axC = fig.add_subplot(gs[1, 0:2])
panel_label(axC, "c", dx=-58)
axC.axvline(0, color=REFERENCE_GREY, lw=0.6, zorder=1)
axC.fill_between(x_grid, line_lo, line_hi, color="0.62", alpha=0.18, linewidth=0, zorder=1)
axC.plot(x_grid, line, color="0.35", lw=1.0, zorder=2)
sc = axC.scatter(focal_contrast, nonfocal_distance, c=CARBON_VALUES, cmap=CMAP_SEQ,
                 norm=matplotlib.colors.LogNorm(vmin=CARBON_VALUES.min(), vmax=CARBON_VALUES.max()),
                 s=26, edgecolor="white", lw=0.6, zorder=3)
cb = fig.colorbar(sc, cax=axC.inset_axes([0.075, 0.895, 0.24, 0.028]), orientation="horizontal")
cb.set_label(CARBON_AXIS_LABEL, fontsize=5.3, labelpad=2)
cb.ax.xaxis.set_label_position("top")
cb.ax.tick_params(labelsize=5.0, length=1.6, pad=1.2)
cb.outline.set_visible(False)
axC.set_xlabel(r"History effect on $\it{A.\ caviae}$ response" "\n(resistance-evolved minus ancestral Δ relative abundance)")
axC.set_ylabel("Divergence of non-focal response between\nhistories at 2 µg ml$^{-1}$ (Aitchison distance)")

# D: species-level responses (Table S9); asterisks mark adjusted P < 0.05.
axD = fig.add_subplot(gs[:, 2])
panel_label(axD, "d", dx=-20)
axD.axvline(0, color=REFERENCE_GREY, lw=0.6)
rng = np.random.default_rng(RANDOM_SEED + 777)
for community, offset, color in [("ANC_SIXTEEN", -0.13, ANCESTRAL_COLOR), ("EVO_SIXTEEN", 0.13, EVOLVED_COLOR)]:
    for yi, sp in enumerate(order):
        vals = rel_rep.loc[rel_rep.community == community, f"d_rel_{sp}"].to_numpy(float)
        axD.scatter(vals, yi + offset + rng.uniform(-0.055, 0.055, len(vals)), s=5.5, facecolor="white", edgecolor=color,
                    lw=0.45, alpha=0.85, zorder=2)
        axD.errorbar(vals.mean(), yi + offset, xerr=1.96 * vals.std(ddof=1) / np.sqrt(len(vals)), fmt="o", mfc=color, mec=color,
                     color=color, ms=3.7, lw=0.9, capsize=1.7, zorder=3)
xmin, xmax = axD.get_xlim()
span = xmax - xmin
for yi, sp in enumerate(order):
    if contrast.loc[sp, "p_adjusted"] < 0.05:
        axD.text(xmax + 0.07 * span, yi, "*", ha="center", va="center", fontsize=8.5, fontweight="bold")
axD.set_xlim(xmin, xmax + 0.14 * span)
axD.set_yticks(np.arange(len(order)))
axD.set_yticklabels([species_plot_label(sp) for sp in order], fontsize=4.4)
axD.tick_params(axis="y", pad=1.5)
focal_row = order.index(FOCAL_SPECIES)
axD.axhspan(focal_row - 0.5, focal_row + 0.5, color="0.90", lw=0, zorder=0)
for tick, sp in zip(axD.get_yticklabels(), order):
    if sp == FOCAL_SPECIES:
        tick.set_fontweight("bold")
axD.set_xlabel("Change in relative abundance\n(0 to 2 µg ml$^{-1}$ streptomycin)")

fig.legend(handles=[
    Line2D([0], [0], marker="o", color=ANCESTRAL_COLOR, mfc=ANCESTRAL_COLOR, mec=ANCESTRAL_COLOR, lw=1.0, label=r"Ancestral $\it{A.\ caviae}$"),
    Line2D([0], [0], marker="o", color=EVOLVED_COLOR, mfc=EVOLVED_COLOR, mec=EVOLVED_COLOR, lw=1.0, label=r"Resistance-evolved $\it{A.\ caviae}$")],
    loc="lower center", bbox_to_anchor=(0.67, 0.010), ncol=2, frameon=False, fontsize=5.2, columnspacing=1.2)
fig.subplots_adjust(top=0.94, bottom=0.19, left=0.08, right=0.99)
save_figure(fig, "Figure_4_low_dose_response")

## Figure S7: species-level CLR responses at every non-zero dose (16-species community)

In [10]:
fig, axes = plt.subplots(1, 3, figsize=(183 * MM, 130 * MM), sharey=True)
for col, dose in enumerate(NONZERO_DOSES):
    ax = axes[col]
    order_clr = list(species_contrasts_clr.loc[species_contrasts_clr.streptomycin == dose]
                     .set_index("species").evolved_minus_ancestral_delta_clr.sort_values().index)
    ax.axvline(0, color="0.7", lw=0.65)
    for community, offset, color in [("ANC_SIXTEEN", -0.12, ANCESTRAL_COLOR), ("EVO_SIXTEEN", 0.12, EVOLVED_COLOR)]:
        for yi, sp in enumerate(order_clr):
            vals = dclr_rep.loc[(dclr_rep.community == community) & (dclr_rep.streptomycin == dose), f"d_clr_{sp}"].to_numpy(float)
            ax.errorbar(vals.mean(), yi + offset, xerr=1.96 * vals.std(ddof=1) / np.sqrt(len(vals)), fmt="o", mfc=color, mec=color,
                        color=color, ms=3.2, lw=0.8, capsize=1.5)
    ax.set_yticks(np.arange(len(order_clr))); ax.set_yticklabels([species_plot_label(sp) for sp in order_clr], fontsize=4.6)
    ax.set_xlabel("ΔCLR relative to no antibiotic")
    ax.text(0.5, 1.025, f"{dose:g} µg ml$^{{-1}}$", transform=ax.transAxes, ha="center", va="bottom", fontsize=6.4)
    if col > 0:
        ax.tick_params(axis="y", labelleft=False)
fig.legend(handles=history_legend_handles(), loc="lower center", bbox_to_anchor=(0.5, 0.01), ncol=2, frameon=False, fontsize=5.4)
fig.subplots_adjust(left=0.21, right=0.99, bottom=0.16, top=0.94, wspace=0.20)
save_figure(fig, "Figure_S7_species_clr_responses")